# Legal AI Agent — Colab Embedding Pipeline

**Mục tiêu:** Chạy embedding 110k+ văn bản pháp luật VN trên GPU Colab, lưu vectorstore về Google Drive.

**Thứ tự chạy:** Chạy từng cell theo thứ tự từ trên xuống.

**Thời gian ước tính:**
- Download data: ~30-45 phút
- Embedding (T4 GPU): ~2-4 giờ
- Build BM25: ~10 phút

## Cell 1 — Kiểm tra GPU

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ Không có GPU! Vào Runtime → Change runtime type → T4 GPU")

## Cell 2 — Mount Google Drive (lưu output)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Thư mục lưu vectorstore + parent_store trên Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/LegalAI_vectorstore'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Output sẽ được lưu tại: {DRIVE_OUTPUT}')

## Cell 3 — Clone repo & cài dependencies

In [ ]:
import os

os.chdir('/content')
!git clone https://github.com/HoangNhatTR/ProjectGenAI_2.git
os.chdir('/content/ProjectGenAI_2')

# Cài dependencies (bỏ neo4j nếu không dùng KG trên Colab)
!pip install -q -r requirements.txt
print('✓ Dependencies installed')

## Cell 4 — Tạo file .env

In [ ]:
import os

# ⚙️ CHỈNH SỬA CÁC GIÁ TRỊ NÀY trước khi chạy
ROUTER9_API_KEY = ""   # API key của bạn (nếu có)
GEMINI_API_KEY  = ""   # Gemini API key (nếu có)

env_content = f"""# Auto-generated for Colab
LLM_PROVIDER=router9
ROUTER9_API_KEY={ROUTER9_API_KEY}
ROUTER9_BASE_URL=http://localhost:20128/v1
ROUTER9_MODEL=cc/claude-haiku-4-5-20251001
GEMINI_API_KEY={GEMINI_API_KEY}

# Embedding — tự động dùng CUDA nếu có GPU
EMBEDDING_MODEL=BAAI/bge-m3

# Vectorstore — lưu vào Drive
VECTORSTORE_DIR=/content/drive/MyDrive/LegalAI_vectorstore/chroma
COLLECTION_NAME=legal_docs

# Chunk settings
CHUNK_SIZE=600
CHUNK_OVERLAP=80
TOP_K=5

# Parent-Child chunking
USE_PARENT_CHILD=true

# HyDE — tắt trong lúc ingest
USE_HYDE=false
"""

with open('/content/ProjectGenAI_2/.env', 'w') as f:
    f.write(env_content)
print('✓ .env created')
!cat /content/ProjectGenAI_2/.env

## Cell 5 — Download dữ liệu từ HuggingFace

In [ ]:
os.chdir('/content/ProjectGenAI_2')

# Kiểm tra disk space trước
!df -h /content

print('\n⏳ Bắt đầu download ~88k văn bản từ HuggingFace...')
print('   Ước tính: 30-45 phút (lần đầu), nhanh hơn nếu đã có cache\n')

!python -m scripts.load_hf_dataset

## Cell 5b — (Tùy chọn) Download thêm dữ liệu vbpl.vn

In [ ]:
# Bỏ comment để chạy nếu muốn crawl thêm từ vbpl.vn
# !python -m scripts.run_supplement
print('Bỏ qua crawl vbpl.vn — dùng HuggingFace data là đủ')

## Cell 6 — Kiểm tra data đã download

In [ ]:
from pathlib import Path

raw_dir = Path('/content/ProjectGenAI_2/data/raw')
total = 0
for folder in sorted(raw_dir.iterdir()):
    if folder.is_dir():
        count = len(list(folder.rglob('*.txt')))
        size_mb = sum(f.stat().st_size for f in folder.rglob('*.txt')) / 1e6
        if count > 0:
            print(f'  {folder.name:25s}: {count:6,} files | {size_mb:6.0f} MB')
            total += count

root_txt = len(list(raw_dir.glob('*.txt')))
print(f'  {"root":25s}: {root_txt:6,} files')
total += root_txt
print(f'\n  TỔNG: {total:,} files')
!df -h /content

## Cell 7 — Patch Embedder để dùng GPU + batch_size lớn hơn

In [ ]:
# Monkey-patch config để dùng GPU batch_size=64 thay vì 16
import sys
sys.path.insert(0, '/content/ProjectGenAI_2')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Patch Embedder.__init__ để set device + batch_size tối ưu cho GPU
from src import embedding as emb_module
from src.schemas import Chunk
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import Optional

original_init = emb_module.Embedder.__init__

def gpu_init(self, model_name, device=None, batch_size=16, max_seq_length=1024):
    self.model_name = model_name
    self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
    self.batch_size = 64 if self.device == 'cuda' else 16  # GPU: 64, CPU: 16
    self.max_seq_length = max_seq_length
    self._model = None
    print(f'  Embedder: device={self.device}, batch_size={self.batch_size}')

emb_module.Embedder.__init__ = gpu_init
print('✓ Embedder patched for GPU (batch_size=64)')

## Cell 8 — Chạy Ingest (Embedding + Vectorstore)

> ⚡ Đây là bước chính — cần GPU. T4 GPU ước tính ~2-4 giờ cho 110k files.

In [ ]:
import os, time
os.chdir('/content/ProjectGenAI_2')

# Ghi đè VECTORSTORE_DIR vào env để lưu vào Drive
os.environ['VECTORSTORE_DIR'] = '/content/drive/MyDrive/LegalAI_vectorstore/chroma'
os.makedirs(os.environ['VECTORSTORE_DIR'], exist_ok=True)

print('🚀 Bắt đầu ingest...')
print(f'   Vectorstore sẽ lưu tại: {os.environ["VECTORSTORE_DIR"]}')
print('   Ước tính: 2-4 giờ trên T4 GPU\n')

start = time.time()
!python -m scripts.ingest --reset
elapsed = (time.time() - start) / 3600
print(f'\n✓ Ingest hoàn thành sau {elapsed:.1f} giờ')

## Cell 9 — Copy Parent Store về Drive

In [ ]:
import shutil

src_db = '/content/ProjectGenAI_2/data/processed/parent_store.db'
dst_db = '/content/drive/MyDrive/LegalAI_vectorstore/parent_store.db'

if os.path.exists(src_db):
    shutil.copy2(src_db, dst_db)
    size_mb = os.path.getsize(dst_db) / 1e6
    print(f'✓ parent_store.db → Drive ({size_mb:.0f} MB)')
else:
    print('⚠ parent_store.db không tìm thấy — kiểm tra USE_PARENT_CHILD=true trong .env')

## Cell 10 — Build BM25 Index

In [ ]:
os.chdir('/content/ProjectGenAI_2')
print('⏳ Building BM25 index...')
!python -m scripts.build_bm25

# Copy BM25 index về Drive
bm25_src = '/content/ProjectGenAI_2/data/bm25'
bm25_dst = '/content/drive/MyDrive/LegalAI_vectorstore/bm25'
if os.path.exists(bm25_src):
    shutil.copytree(bm25_src, bm25_dst, dirs_exist_ok=True)
    size_mb = sum(f.stat().st_size for f in Path(bm25_dst).rglob('*') if f.is_file()) / 1e6
    print(f'✓ BM25 index → Drive ({size_mb:.0f} MB)')

## Cell 11 — Verify & Thống kê kết quả

In [ ]:
import sys
sys.path.insert(0, '/content/ProjectGenAI_2')
from dotenv import load_dotenv
load_dotenv('/content/ProjectGenAI_2/.env')

from src import config
from src.vectorstore import VectorStore
from src.parent_store import ParentStore

store = VectorStore(config.VECTORSTORE_DIR, config.COLLECTION_NAME)
chunk_count = store.count()
print(f'✓ Vectorstore: {chunk_count:,} chunks')

ps_path = '/content/drive/MyDrive/LegalAI_vectorstore/parent_store.db'
if os.path.exists(ps_path):
    ps = ParentStore(ps_path)
    print(f'✓ Parent store: {ps.count():,} parents')

bm25_path = Path('/content/drive/MyDrive/LegalAI_vectorstore/bm25')
if bm25_path.exists():
    bm25_size = sum(f.stat().st_size for f in bm25_path.rglob('*') if f.is_file()) / 1e6
    print(f'✓ BM25 index: {bm25_size:.0f} MB')

print('\n=== FILES TRÊN DRIVE ===')
!ls -lh /content/drive/MyDrive/LegalAI_vectorstore/

## Cell 12 — (Tùy chọn) Test nhanh RAG

In [ ]:
from src.embedding import Embedder
from src.retriever import Retriever
from src.bm25_index import BM25Index

embedder = Embedder(config.EMBEDDING_MODEL)
store_obj = VectorStore(config.VECTORSTORE_DIR, config.COLLECTION_NAME)
bm25 = BM25Index()
# bm25.load('/content/ProjectGenAI_2/data/bm25')  # uncomment nếu BM25 đã build

retriever = Retriever(embedder=embedder, store=store_obj, bm25=bm25)

test_query = 'Mức phạt vượt đèn đỏ xe máy là bao nhiêu?'
results = retriever.retrieve(test_query, top_k=3, use_kg=False)

print(f'Query: {test_query}\n')
for i, r in enumerate(results, 1):
    meta = r.chunk.metadata
    loc = ' - '.join(filter(None, [r.chunk.article, r.chunk.clause, r.chunk.point]))
    print(f'[{i}] score={r.score:.3f} | {meta.doc_number or meta.title} | {loc}')
    print(f'     {r.chunk.text[:150]}...')
    print()

## Hướng dẫn dùng output trên server

Sau khi notebook chạy xong, trên Google Drive bạn sẽ có:
```
LegalAI_vectorstore/
├── chroma/          ← Vectorstore Chroma (lớn nhất)
├── parent_store.db  ← SQLite parent chunks
└── bm25/            ← BM25 keyword index
```

### Cách deploy lên server

```bash
# 1. Clone code
git clone https://github.com/HoangNhatTR/ProjectGenAI_2.git
cd ProjectGenAI_2

# 2. Tải vectorstore từ Google Drive về server
#    (dùng rclone, gdown, hoặc drive API)
rclone copy drive:LegalAI_vectorstore ./data/

# 3. Config .env
cp .env.example .env
# Chỉnh VECTORSTORE_DIR=./data/chroma

# 4. Chạy app
python api.py
```